# Voice Preprocessing: Person + Language Classification

This notebook prepares the feature dataset for a model that predicts **which person is speaking** and **which language** they are speaking (Arabic, English, German, French). Each original recording is kept as-is, and several **augmented copies** (noise, pitch shift, time stretch, time shift, volume change) are generated from it to help the model generalize to new recording conditions.

Assumed folder structure (all files already `.wav`, no format conversion needed):
```
Data/
  person_1/
    Arabic/
      voice1.wav
      voice2.wav
    English/
      ...
    German/
      ...
    French/
      ...
  person_2/
    Arabic/
    ...
```
If your folders are organized differently (e.g. `Data/<language>/<person>/`, or a flat folder with `person_language_i.wav` filenames), the loading cell below will need small changes — let me know and I can adapt it.

## Imports
Libraries used for audio loading, feature extraction, and data handling.

In [1]:
import librosa
import numpy as np
import os
import pandas as pd


## Data Augmentation

To help the model generalize (different microphones, background noise, speaking speed, phone quality, etc.), each original recording is kept **as-is**, and a few augmented copies are additionally generated from it. Every augmented copy keeps the same `Person` and `Language` label as the original — only the audio itself is perturbed:

- **Add noise** — mixes in a small amount of random (white) noise, simulating background/mic noise.
- **Time stretch** — speeds up or slows down the speech slightly without changing pitch, simulating faster/slower talkers.
- **Pitch shift** — raises or lowers the pitch a bit, simulating natural voice pitch variation.
- **Time shift** — shifts the audio slightly earlier/later in time (circular shift), simulating imperfect trimming/timing.
- **Volume change** — scales the amplitude up or down, simulating recording at different distances/volumes.

You can toggle which augmentations are used and how many augmented copies per file via `AUGMENTATIONS` and `N_AUGMENTED_COPIES` below.

In [2]:
def add_noise(y, noise_factor=0.005):
    noise = np.random.randn(len(y))
    return y + noise_factor * noise


def time_stretch(y, rate=None):
    if rate is None:
        rate = np.random.uniform(0.85, 1.15)   # 15% slower to 15% faster
    return librosa.effects.time_stretch(y=y, rate=rate)


def pitch_shift(y, sr, n_steps=None):
    if n_steps is None:
        n_steps = np.random.uniform(-2, 2)     # +/- 2 semitones
    return librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)


def time_shift(y, shift_max=0.2):
    shift_amount = int(len(y) * np.random.uniform(-shift_max, shift_max))
    return np.roll(y, shift_amount)


def change_volume(y, gain_range=(0.6, 1.4)):
    gain = np.random.uniform(*gain_range)
    return y * gain


# Which augmentations to apply, and how many augmented copies to make per original file.
# Each copy applies ONE randomly-picked augmentation from this list (kept simple/independent
# so it's clear which effect produced which copy). Set to [] to disable augmentation entirely.
AUGMENTATIONS = ['noise', 'time_stretch', 'pitch_shift', 'time_shift', 'volume']
N_AUGMENTED_COPIES = 3   # how many augmented versions to generate per original recording


def augment_audio(y, sr, augmentation_name):
    if augmentation_name == 'noise':
        return add_noise(y)
    elif augmentation_name == 'time_stretch':
        return time_stretch(y)
    elif augmentation_name == 'pitch_shift':
        return pitch_shift(y, sr)
    elif augmentation_name == 'time_shift':
        return time_shift(y)
    elif augmentation_name == 'volume':
        return change_volume(y)
    else:
        raise ValueError(f"Unknown augmentation: {augmentation_name}")


## Extract Features

For every `.wav` file we build one fixed-length feature vector combining several complementary descriptors:

- **MFCCs (17 coeffs)** — timbre / phonetic content, the main cue for both speaker identity and language.
- **Delta MFCCs** — how the MFCCs change over time (captures speech rhythm/articulation speed, which differs a lot between languages).
- **Chroma (12 bins)** — pitch-class energy distribution, adds tonal/intonation info.
- **Spectral contrast (7 bands)** — difference between peaks and valleys in the spectrum, helps distinguish voice qualities between speakers.
- **Spectral centroid** — "brightness" of the voice, a simple speaker-timbre cue.
- **Zero-crossing rate** — roughly captures how noisy/percussive the signal is, useful for consonant-heavy vs. vowel-heavy languages (e.g. Arabic/German vs. French).

Steps per file:
1. Load the audio at a fixed sample rate.
2. Trim leading/trailing silence.
3. Keep the **original** (untouched) trimmed audio, and additionally generate `N_AUGMENTED_COPIES` **augmented** versions of it.
4. Extract each feature above from every version (original + augmented copies).
5. Normalize and average each over time, then concatenate into one vector per version.
6. Record the `Person` label (from the parent folder), `Language` label (from the sub-folder), and an `Augmentation` label (`'original'` or the augmentation name) for every version.

In [3]:
data_path = "./Dataset"
SR = 22050           # sample rate to resample every file to
N_MFCC = 17           # number of MFCC coefficients
N_CHROMA = 12         # number of chroma bins
N_CONTRAST = 7        # number of spectral contrast bands
TOP_DB = 30           # silence threshold for trimming


def extract_features(y, sr):
    """Build one fixed-length feature vector for a single (already-trimmed) audio signal."""

    # --- MFCCs + deltas: phonetic/timbre content and how it changes over time ---
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=N_MFCC)
    mfcc = librosa.util.normalize(mfcc)
    mfcc_delta = librosa.feature.delta(mfcc)

    # --- Chroma: pitch-class / tonal content ---
    chroma = librosa.feature.chroma_stft(y=y, sr=sr, n_chroma=N_CHROMA)

    # --- Spectral contrast: peak-vs-valley energy, useful speaker cue ---
    contrast = librosa.feature.spectral_contrast(y=y, sr=sr, n_bands=N_CONTRAST - 1)

    # --- Spectral centroid: brightness of the voice ---
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

    # --- Zero-crossing rate: noisiness / consonant vs. vowel balance ---
    zcr = librosa.feature.zero_crossing_rate(y=y)

    # Collapse every feature's time axis to its mean, then concatenate into one vector
    feature_vector = np.concatenate([
        np.mean(mfcc, axis=1),
        np.mean(mfcc_delta, axis=1),
        np.mean(chroma, axis=1),
        np.mean(contrast, axis=1),
        np.mean(centroid, axis=1),
        np.mean(zcr, axis=1),
    ])
    return feature_vector


Person = []
Language = []
Augmentation = []
Features = []

for person in os.listdir(data_path):
    person_path = os.path.join(data_path, person)
    if not os.path.isdir(person_path):
        continue

    for language in os.listdir(person_path):
        language_path = os.path.join(person_path, language)
        if not os.path.isdir(language_path):
            continue

        for voice in os.listdir(language_path):
            if not voice.endswith('.wav'):
                continue

            audio_path = os.path.join(language_path, voice)

            librosa_audio, sr = librosa.load(audio_path, sr=SR)              # load audio file
            removed_silence = librosa.effects.trim(librosa_audio, top_db=TOP_DB)[0]  # remove silence

            # --- 1) keep the original, untouched recording ---
            feature_vector = extract_features(removed_silence, SR)
            Person.append(person)
            Language.append(language)
            Augmentation.append('original')
            Features.append(feature_vector)

            # --- 2) add N_AUGMENTED_COPIES augmented versions of the same recording ---
            for i in range(N_AUGMENTED_COPIES):
                augmentation_name = np.random.choice(AUGMENTATIONS)
                try:
                    augmented_audio = augment_audio(removed_silence, SR, augmentation_name)
                except Exception as e:
                    print(f"Skipped augmentation '{augmentation_name}' on {audio_path}: {e}")
                    continue

                augmented_feature_vector = extract_features(augmented_audio, SR)
                Person.append(person)
                Language.append(language)
                Augmentation.append(augmentation_name)
                Features.append(augmented_feature_vector)

Features = np.array(Features)
Person = np.array(Person)
Language = np.array(Language)
Augmentation = np.array(Augmentation)

print("Features shape:", Features.shape)
print("Person shape:", Person.shape)
print("Language shape:", Language.shape)
print("Languages found:", np.unique(Language))
print("Persons found:", np.unique(Person))
print("Augmentation counts:", dict(zip(*np.unique(Augmentation, return_counts=True))))


Features shape: (1220, 55)
Person shape: (1220,)
Language shape: (1220,)
Languages found: ['Arabic' 'English' 'French' 'German']
Persons found: ['EsraaM' 'MWalaa' 'MariamB']
Augmentation counts: {np.str_('noise'): np.int64(167), np.str_('original'): np.int64(305), np.str_('pitch_shift'): np.int64(186), np.str_('time_shift'): np.int64(178), np.str_('time_stretch'): np.int64(183), np.str_('volume'): np.int64(201)}


## Save the Preprocessed Dataset
Store the feature matrix and labels so the next notebook (model training) can load them directly, without re-running audio processing every time.

In [4]:
feature_names = (
    [f"mfcc_{i+1}" for i in range(N_MFCC)] +
    [f"mfcc_delta_{i+1}" for i in range(N_MFCC)] +
    [f"chroma_{i+1}" for i in range(N_CHROMA)] +
    [f"contrast_{i+1}" for i in range(N_CONTRAST)] +
    ["spectral_centroid"] +
    ["zero_crossing_rate"]
)

df = pd.DataFrame(Features, columns=feature_names)
df["Person"] = Person
df["Language"] = Language
df["Augmentation"] = Augmentation

df.to_csv("preprocessed_features.csv", index=False)
print("Saved preprocessed_features.csv with shape:", df.shape)


Saved preprocessed_features.csv with shape: (1220, 58)
